In [1]:
import torch

print("GPU disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU disponível: True
GPU: Tesla T4


In [2]:
import pandas as pd

df = pd.read_excel("/content/train.xlsx")

print("Quantidade de linhas:", len(df))
print("\nColunas:")
print(df.columns.tolist())

print("\nDistribuição das classes:")
print(df["clarity"].value_counts())

df.head()

Quantidade de linhas: 20092

Colunas:
['resp_text', 'clarity']

Distribuição das classes:
clarity
c5      6892
c234    6853
c1      6347
Name: count, dtype: int64


,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234


In [3]:
from sklearn.model_selection import train_test_split

X = df["resp_text"].fillna("").astype(str)
y = df["clarity"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treinamento:", len(X_train))
print("Validação:", len(X_val))

print("\nDistribuição no treinamento:")
print(y_train.value_counts())

print("\nDistribuição na validação:")
print(y_val.value_counts())

Treinamento: 16073
Validação: 4019

Distribuição no treinamento:
clarity
c5      5513
c234    5482
c1      5078
Name: count, dtype: int64

Distribuição na validação:
clarity
c5      1379
c234    1371
c1      1269
Name: count, dtype: int64


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

baseline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            sublinear_tf=True,
            max_features=100000
        )
    ),
    (
        "logreg",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

print("Treinando baseline...")

baseline.fit(X_train, y_train)

print("Treinamento concluído!")

Treinando baseline...
Treinamento concluído!


In [5]:
pred_baseline = baseline.predict(X_val)

acc_baseline = accuracy_score(y_val, pred_baseline)
f1_baseline = f1_score(y_val, pred_baseline, average="macro")

print("RESULTADO DO BASELINE")
print("=" * 50)

print(f"Acurácia: {acc_baseline:.4f}")
print(f"Acurácia (%): {acc_baseline * 100:.2f}%")
print(f"F1 Macro: {f1_baseline:.4f}")

print("\nRelatório por classe:")
print(classification_report(y_val, pred_baseline, digits=4))

print("\nMatriz de confusão:")
print(
    confusion_matrix(
        y_val,
        pred_baseline,
        labels=["c1", "c234", "c5"]
    )
)

RESULTADO DO BASELINE
Acurácia: 0.4499
Acurácia (%): 44.99%
F1 Macro: 0.4474

Relatório por classe:
              precision    recall  f1-score   support

          c1     0.4726    0.4350    0.4530      1269
        c234     0.3943    0.3632    0.3781      1371
          c5     0.4773    0.5497    0.5110      1379

    accuracy                         0.4499      4019
   macro avg     0.4481    0.4493    0.4474      4019
weighted avg     0.4475    0.4499    0.4474      4019


Matriz de confusão:
[[552 402 315]
 [358 498 515]
 [258 363 758]]


In [6]:
!pip install -q transformers

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Nenhuma")

Dispositivo: cuda
GPU: Tesla T4


In [8]:
label2id = {
    "c1": 0,
    "c234": 1,
    "c5": 2
}

id2label = {
    0: "c1",
    1: "c234",
    2: "c5"
}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

model = model.to(device)

print("BERTimbau carregado!")

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

BERTimbau carregado!


In [9]:
class TextDataset(Dataset):

    def __init__(self, textos, labels):
        self.textos = textos.tolist()
        self.labels = [label2id[label] for label in labels.tolist()]

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):

        encoding = tokenizer(
            self.textos[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }


train_dataset = TextDataset(
    X_train,
    y_train
)

val_dataset = TextDataset(
    X_val,
    y_val
)

print("Treino BERT:", len(train_dataset))
print("Validação BERT:", len(val_dataset))

Treino BERT: 16073
Validação BERT: 4019


In [10]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Batches de treinamento:", len(train_loader))
print("Batches de validação:", len(val_loader))

Batches de treinamento: 1005
Batches de validação: 252


In [11]:
from torch.optim import AdamW
from tqdm.auto import tqdm

EPOCHS = 3
LEARNING_RATE = 2e-5

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

for epoch in range(EPOCHS):

    print(f"\n===== ÉPOCA {epoch + 1}/{EPOCHS} =====")

    # -------------------------
    # TREINAMENTO
    # -------------------------

    model.train()

    total_loss = 0

    barra = tqdm(
        train_loader,
        desc="Treinando"
    )

    for batch in barra:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

        barra.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    media_loss = total_loss / len(train_loader)

    print(
        f"Loss médio do treinamento: {media_loss:.4f}"
    )

    # -------------------------
    # VALIDAÇÃO
    # -------------------------

    model.eval()

    previsoes = []
    valores_reais = []

    with torch.no_grad():

        barra_val = tqdm(
            val_loader,
            desc="Validando"
        )

        for batch in barra_val:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            predicoes = torch.argmax(
                outputs.logits,
                dim=1
            )

            previsoes.extend(
                predicoes.cpu().numpy()
            )

            valores_reais.extend(
                batch["labels"].numpy()
            )

    acc = accuracy_score(
        valores_reais,
        previsoes
    )

    f1 = f1_score(
        valores_reais,
        previsoes,
        average="macro"
    )

    print(f"\nAcurácia validação: {acc * 100:.2f}%")
    print(f"F1 Macro validação: {f1:.4f}")


===== ÉPOCA 1/3 =====


Treinando:   0%|          | 0/1005 [00:00<?, ?it/s]

Loss médio do treinamento: 1.0684


Validando:   0%|          | 0/252 [00:00<?, ?it/s]


Acurácia validação: 44.61%
F1 Macro validação: 0.4144

===== ÉPOCA 2/3 =====


...


Acurácia validação: 44.91%
F1 Macro validação: 0.4420

===== ÉPOCA 3/3 =====


Treinando:   0%|          | 0/1005 [00:00<?, ?it/s]

Loss médio do treinamento: 0.8983


Validando:   0%|          | 0/252 [00:00<?, ?it/s]


Acurácia validação: 44.51%
F1 Macro validação: 0.4458


In [12]:
from tqdm.auto import tqdm
import numpy as np

tamanhos = []

for texto in tqdm(X_train, desc="Analisando textos"):
    tokens = tokenizer(
        texto,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]

    tamanhos.append(len(tokens))

print("Média:", np.mean(tamanhos))
print("Mediana:", np.median(tamanhos))
print("75%:", np.percentile(tamanhos, 75))
print("90%:", np.percentile(tamanhos, 90))
print("95%:", np.percentile(tamanhos, 95))
print("Máximo:", np.max(tamanhos))

print("\nTextos maiores que 128:",
      np.mean(np.array(tamanhos) > 128) * 100, "%")

print("Textos maiores que 256:",
      np.mean(np.array(tamanhos) > 256) * 100, "%")

print("Textos maiores que 512:",
      np.mean(np.array(tamanhos) > 512) * 100, "%")

Analisando textos:   0%|          | 0/16073 [00:00<?, ?it/s]

Média: 253.1215081192061
Mediana: 182.0
75%: 331.0
90%: 535.0
95%: 718.0
Máximo: 4429

Textos maiores que 128: 63.20537547439806 %
Textos maiores que 256: 35.77427984819262 %
Textos maiores que 512: 10.844273004417346 %


In [13]:
import torch
import copy
import numpy as np

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.metrics import accuracy_score, f1_score

# ==============================
# CONFIGURAÇÕES BERTIMBAU V2
# ==============================

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dispositivo:", device)

Dispositivo: cuda


In [16]:
# ==============================
# RÓTULOS
# ==============================

label2id = {
    "c1": 0,
    "c234": 1,
    "c5": 2
}

id2label = {
    0: "c1",
    1: "c234",
    2: "c5"
}


# ==============================
# DATASET
# ==============================

class TextDatasetV2(Dataset):

    def __init__(self, textos, labels):
        self.textos = textos.tolist()
        self.labels = [
            label2id[label]
            for label in labels.tolist()
        ]

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):

        encoding = tokenizer(
            self.textos[idx],

            truncation=True,
            padding="max_length",

            # PRINCIPAL ALTERAÇÃO
            max_length=MAX_LENGTH,

            return_tensors="pt"
        )

        return {
            "input_ids":
                encoding["input_ids"].squeeze(0),

            "attention_mask":
                encoding["attention_mask"].squeeze(0),

            "labels":
                torch.tensor(
                    self.labels[idx],
                    dtype=torch.long
                )
        }

In [17]:
train_dataset_v2 = TextDatasetV2(
    X_train,
    y_train
)

val_dataset_v2 = TextDatasetV2(
    X_val,
    y_val
)


train_loader_v2 = DataLoader(
    train_dataset_v2,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader_v2 = DataLoader(
    val_dataset_v2,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Treino:", len(train_dataset_v2))
print("Validação:", len(val_dataset_v2))

print("Batches treino:", len(train_loader_v2))
print("Batches validação:", len(val_loader_v2))

Treino: 16073
Validação: 4019
Batches treino: 2010
Batches validação: 503


In [18]:
model_v2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

model_v2 = model_v2.to(device)

print("Novo BERTimbau carregado!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Novo BERTimbau carregado!


In [19]:
optimizer = torch.optim.AdamW(
    model_v2.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

total_steps = (
    len(train_loader_v2)
    * EPOCHS
)

warmup_steps = int(
    total_steps * 0.10
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("Total de passos:", total_steps)
print("Warmup:", warmup_steps)

Total de passos: 6030
Warmup: 603


In [20]:
from tqdm.auto import tqdm

melhor_acuracia = 0
melhor_epoca = 0
melhor_estado = None

historico = []

for epoch in range(EPOCHS):

    print("\n" + "=" * 60)
    print(f"ÉPOCA {epoch + 1}/{EPOCHS}")
    print("=" * 60)

    # ==============================
    # TREINAMENTO
    # ==============================

    model_v2.train()

    total_loss = 0

    barra = tqdm(
        train_loader_v2,
        desc="Treinando"
    )

    for batch in barra:

        optimizer.zero_grad()

        input_ids = (
            batch["input_ids"]
            .to(device)
        )

        attention_mask = (
            batch["attention_mask"]
            .to(device)
        )

        labels = (
            batch["labels"]
            .to(device)
        )

        outputs = model_v2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model_v2.parameters(),
            1.0
        )

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        barra.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    media_loss = (
        total_loss /
        len(train_loader_v2)
    )


    # ==============================
    # VALIDAÇÃO
    # ==============================

    model_v2.eval()

    previsoes = []
    reais = []

    with torch.no_grad():

        for batch in tqdm(
            val_loader_v2,
            desc="Validando"
        ):

            input_ids = (
                batch["input_ids"]
                .to(device)
            )

            attention_mask = (
                batch["attention_mask"]
                .to(device)
            )

            outputs = model_v2(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(
                outputs.logits,
                dim=1
            )

            previsoes.extend(
                preds.cpu().numpy()
            )

            reais.extend(
                batch["labels"].numpy()
            )


    acc = accuracy_score(
        reais,
        previsoes
    )

    f1 = f1_score(
        reais,
        previsoes,
        average="macro"
    )


    print(
        f"\nLoss médio: {media_loss:.4f}"
    )

    print(
        f"Acurácia: {acc * 100:.2f}%"
    )

    print(
        f"F1 Macro: {f1:.4f}"
    )


    historico.append({
        "epoca": epoch + 1,
        "loss": media_loss,
        "acuracia": acc,
        "f1_macro": f1
    })


    # ==============================
    # SALVAR MELHOR ÉPOCA
    # ==============================

    if acc > melhor_acuracia:

        melhor_acuracia = acc
        melhor_epoca = epoch + 1

        melhor_estado = copy.deepcopy(
            model_v2.state_dict()
        )

        print(
            "\n*** NOVO MELHOR MODELO ***"
        )


ÉPOCA 1/3


Treinando:   0%|          | 0/2010 [00:00<?, ?it/s]

Validando:   0%|          | 0/503 [00:00<?, ?it/s]


Loss médio: 1.0726
Acurácia: 44.79%
F1 Macro: 0.4206

*** NOVO MELHOR MODELO ***

ÉPOCA 2/3


Treinando:   0%|          | 0/2010 [00:00<?, ?it/s]

Validando:   0%|          | 0/503 [00:00<?, ?it/s]


Loss médio: 0.9994
Acurácia: 45.91%
F1 Macro: 0.4489

*** NOVO MELHOR MODELO ***

ÉPOCA 3/3


Treinando:   0%|          | 0/2010 [00:00<?, ?it/s]

Validando:   0%|          | 0/503 [00:00<?, ?it/s]


Loss médio: 0.8713
Acurácia: 45.93%
F1 Macro: 0.4561

*** NOVO MELHOR MODELO ***


In [21]:
model_v2.load_state_dict(melhor_estado)

print(f"Melhor época restaurada: {melhor_epoca}")
print(f"Melhor acurácia: {melhor_acuracia * 100:.2f}%")

Melhor época restaurada: 3
Melhor acurácia: 45.93%


In [22]:
PASTA_MODELO = "/content/bertimbau_v2_melhor"

model_v2.save_pretrained(PASTA_MODELO)
tokenizer.save_pretrained(PASTA_MODELO)

print("Modelo salvo em:", PASTA_MODELO)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo salvo em: /content/bertimbau_v2_melhor


In [23]:
X_full = df["resp_text"].fillna("").astype(str)
y_full = df["clarity"]

full_dataset = TextDatasetV2(
    X_full,
    y_full
)

full_loader = DataLoader(
    full_dataset,
    batch_size=8,
    shuffle=True
)

print("Total usado no treinamento final:", len(full_dataset))
print("Quantidade de batches:", len(full_loader))

Total usado no treinamento final: 20092
Quantidade de batches: 2512


In [24]:
model_final = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

model_final = model_final.to(device)

print("BERTimbau final criado!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

BERTimbau final criado!


In [25]:
EPOCHS_FINAL = 3

optimizer_final = torch.optim.AdamW(
    model_final.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

total_steps_final = (
    len(full_loader) * EPOCHS_FINAL
)

warmup_steps_final = int(
    total_steps_final * 0.10
)

scheduler_final = get_linear_schedule_with_warmup(
    optimizer_final,
    num_warmup_steps=warmup_steps_final,
    num_training_steps=total_steps_final
)

print("Épocas:", EPOCHS_FINAL)
print("Total de passos:", total_steps_final)
print("Warmup:", warmup_steps_final)

Épocas: 3
Total de passos: 7536
Warmup: 753


In [26]:
for epoch in range(EPOCHS_FINAL):

    print("\n" + "=" * 60)
    print(f"TREINAMENTO FINAL - ÉPOCA {epoch + 1}/{EPOCHS_FINAL}")
    print("=" * 60)

    model_final.train()

    total_loss = 0

    barra = tqdm(
        full_loader,
        desc="Treinando"
    )

    for batch in barra:

        optimizer_final.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_final(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model_final.parameters(),
            1.0
        )

        optimizer_final.step()
        scheduler_final.step()

        total_loss += loss.item()

        barra.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    media_loss = total_loss / len(full_loader)

    print(
        f"\nLoss médio da época: {media_loss:.4f}"
    )


TREINAMENTO FINAL - ÉPOCA 1/3


Treinando:   0%|          | 0/2512 [00:00<?, ?it/s]


Loss médio da época: 1.0688

TREINAMENTO FINAL - ÉPOCA 2/3


Treinando:   0%|          | 0/2512 [00:00<?, ?it/s]


Loss médio da época: 0.9977

TREINAMENTO FINAL - ÉPOCA 3/3


Treinando:   0%|          | 0/2512 [00:00<?, ?it/s]


Loss médio da época: 0.8854


In [27]:
PASTA_FINAL = "/content/bertimbau_modelo_final"

model_final.save_pretrained(PASTA_FINAL)
tokenizer.save_pretrained(PASTA_FINAL)

print("MODELO FINAL SALVO!")
print("Local:", PASTA_FINAL)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MODELO FINAL SALVO!
Local: /content/bertimbau_modelo_final


In [28]:
import pandas as pd

test = pd.read_excel("/content/test1.xlsx")

print("Quantidade de linhas:", len(test))
print("Colunas:", test.columns.tolist())

test.head()

Quantidade de linhas: 900
Colunas: ['resp_text', 'clarity']


,resp_text,clarity
0,"Prezada Hellen, Dentre as pesquisas domici...",NaN
1,"Senhor Francisco, O Serviço de Informações a...",NaN
2,"Prezada Srª. Raylane, Em resposta a sua solic...",NaN
3,"Prezada Senhora, De acordo com o art.13 do D...",NaN
4,"Prezado(a) Senhor(a), Em atendimento ao ped...",NaN


In [29]:
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader

# Textos do teste real
textos_test = test["resp_text"].fillna("").astype(str).tolist()


class TestDataset(Dataset):

    def __init__(self, textos):
        self.textos = textos

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):

        encoding = tokenizer(
            self.textos[idx],
            truncation=True,
            padding="max_length",
            max_length=256,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }


test_dataset = TestDataset(textos_test)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

print("Textos para classificar:", len(test_dataset))
print("Batches:", len(test_loader))

Textos para classificar: 900
Batches: 113


In [30]:
model_final.eval()

previsoes_test = []

with torch.no_grad():

    for batch in tqdm(
        test_loader,
        desc="Classificando test1.xlsx"
    ):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model_final(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(
            outputs.logits,
            dim=1
        )

        previsoes_test.extend(
            preds.cpu().numpy()
        )

print("Quantidade de previsões:", len(previsoes_test))

Classificando test1.xlsx:   0%|          | 0/113 [00:00<?, ?it/s]

Quantidade de previsões: 900


In [31]:
previsoes_labels = [
    id2label[int(pred)]
    for pred in previsoes_test
]

test["clarity"] = previsoes_labels

print(test["clarity"].value_counts())

test.head(10)

clarity
c234    349
c5      303
c1      248
Name: count, dtype: int64


,resp_text,clarity
0,"Prezada Hellen, Dentre as pesquisas domici...",c234
1,"Senhor Francisco, O Serviço de Informações a...",c234
2,"Prezada Srª. Raylane, Em resposta a sua solic...",c5
3,"Prezada Senhora, De acordo com o art.13 do D...",c1
4,"Prezado(a) Senhor(a), Em atendimento ao ped...",c234
5,"Prezada senhora, o Serviço de Informações ao ...",c1
6,A informação solicitada encontra-se na página...,c5
7,"Prezado (a) Cidadão (ã), 1.Conforme solicit...",c234
8,"Prezado, bom dia! Informamos que a vossa soli...",c234
9,"Prezado Ssenhor, Em atendimento a sua soli...",c1


In [32]:
print("Linhas:", len(test))
print("Colunas:", test.columns.tolist())
print("Valores vazios em clarity:", test["clarity"].isna().sum())
print("Classes encontradas:", test["clarity"].unique())

print("\nDistribuição:")
print(test["clarity"].value_counts())

Linhas: 900
Colunas: ['resp_text', 'clarity']
Valores vazios em clarity: 0
Classes encontradas: ['c234' 'c5' 'c1']

Distribuição:
clarity
c234    349
c5      303
c1      248
Name: count, dtype: int64


In [33]:
ARQUIVO_SAIDA = "/content/test1_classificado.xlsx"

test.to_excel(
    ARQUIVO_SAIDA,
    index=False
)

print("Arquivo criado!")
print(ARQUIVO_SAIDA)

Arquivo criado!
/content/test1_classificado.xlsx
